In [1]:
import sys, os
sys.path.insert(0, '../../utils')

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.ticker import PercentFormatter
from matplotlib.lines import Line2D
from utils import load_neurons_table, load_synapses_position_transformed
from connectome_types import CONNECTOME_SYN_TABLE_PATH, CONNECTOME_PRE_SYN_TABLE_PATH, SPINE_TABLE_OUTGOING
from neuron_custom_features import calc_spines_features
from spines_utils import filter_valid_neuron_w_spines, split_syn_mat_by_type_four
from plot_utils import ex_color, inh_color, plot_skeleton_continuous
from spine_pref_utils import per_neuron_spine_ratio
from sk_load_utils import load_col_skeleton_only
from figures_utils import add_panel_label, plot_nested_cylinders
from axon_pref_utils import fit_pop, fit_single_wrap, plot_raw_bin, plot_single_neuron

In [3]:
neurons_df = load_neurons_table()
syn_df = load_synapses_position_transformed(base_syn_table_path=CONNECTOME_SYN_TABLE_PATH)
df, syn_with_tags = calc_spines_features(neurons_df, syn_df)
neuron_clf_type = df[['root_id', 'clf_type']].set_index('root_id').to_dict(orient='index')
df, filtered_syn_mat, filtered_bin_mat, filtered_mapping, filtered_reverse_mapping, ex_neurons, inh_neurons = filter_valid_neuron_w_spines(df)

spine_df_outgoing = pd.read_csv(SPINE_TABLE_OUTGOING)
outgoing_syn_df = load_synapses_position_transformed(base_syn_table_path=CONNECTOME_PRE_SYN_TABLE_PATH)
outgoing_syn_with_tags = outgoing_syn_df[outgoing_syn_df.id_.isin(spine_df_outgoing.target_id)].copy()
outgoing_syn_with_tags['tag'] = outgoing_syn_with_tags.id_.map(spine_df_outgoing.set_index('target_id').tag)

ex_outgoing_syn_with_tags = outgoing_syn_with_tags[outgoing_syn_with_tags.pre_clf_type == 'E']

EE, EI, IE, II, ex_idx, inh_idx = split_syn_mat_by_type_four(filtered_bin_mat, filtered_mapping, neuron_clf_type)

ex_syn_tags = syn_with_tags[syn_with_tags.pre_clf_type == 'E']
ee_syn_tags = ex_syn_tags[ex_syn_tags.post_clf_type == 'E']
ex_neurons = ex_neurons.copy()

connectome neurons table:  1351
valid neurons w position: 1351
spine table incoming size (4567647, 4)
spine table outgoing size (819832, 4)
neurons: 1351
synapses with tags: 145356


100%|██████████| 1351/1351 [00:17<00:00, 78.87it/s]


Filtering neurons with valid spine data...
Remaining neurons after filtering: 1298
fixing networks
Filtering: Reducing matrix from 1351 to 1298 neurons.


In [4]:
main_feature = 'spine'

ee_stats = per_neuron_spine_ratio(ee_syn_tags, ex_neurons.root_id.tolist())
e_all_outside_stats = per_neuron_spine_ratio(spine_df_outgoing, ex_neurons.root_id.tolist(), group_by='pre_pt_root_id')

ex_neurons['x_EE'] = ex_neurons.root_id.map(ee_stats['n_syn'])
ex_neurons['x_all_outside'] = ex_neurons.root_id.map(e_all_outside_stats['n_syn'])
ex_neurons[f'outgoing_{main_feature}_ratio_EE'] = ex_neurons.root_id.map(ee_stats['ratio'])
ex_neurons[f'outgoing_{main_feature}_ratio_all_outside'] = ex_neurons.root_id.map(e_all_outside_stats['ratio'])

In [5]:
plt.rcParams['font.size'] = 13
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['font.family'] = 'Arial'

spiny_color  = '#7C3AED'
aspiny_color = '#059669'
spiny_color_ei  = '#BE98FF'
aspiny_color_ei = '#2CFFB1'

titles = ['E \u2192 All', 'E \u2192 E & I', 'E \u2192 E']

In [6]:
def plot_legend_EI_clf_type(ax, labels=['E', 'I'], colors=[ex_color, inh_color],
                             fontsize='small', markersize=6):
    legend_elements = [
        Line2D([0], [0], marker='o', color=colors[i], markerfacecolor=colors[i],
               linestyle='None', markersize=markersize, label=labels[i])
        for i in range(len(labels))
    ]
    ax.legend(handles=legend_elements, frameon=False, title='',
              loc='lower center', bbox_to_anchor=(0.5, 1.1), ncol=2, fontsize=fontsize)
    ax.set_axis_off()

In [7]:
node_id = 864691135617152361
sk_dent = load_col_skeleton_only(node_id, reset_axon=True, old=True)
sk_axon = load_col_skeleton_only(node_id, reset_dentrites=True, old=True)

outgoing_syn_outside = ex_outgoing_syn_with_tags[ex_outgoing_syn_with_tags.pre_id == node_id]
outgoing_syn_all = ex_syn_tags[ex_syn_tags.pre_id == node_id]
outgoing_syn_ee = ee_syn_tags[ee_syn_tags.pre_id == node_id]

outgoing_syn_onto_spines_outside = outgoing_syn_outside[outgoing_syn_outside.tag == 'spine']
outgoing_syn_onto_nonspines_outside = outgoing_syn_outside[outgoing_syn_outside.tag != 'spine']
outgoing_syn_onto_spines_all = outgoing_syn_all[outgoing_syn_all.tag == 'spine']
outgoing_syn_onto_nonspines_all = outgoing_syn_all[outgoing_syn_all.tag != 'spine']
outgoing_syn_onto_spines_ee = outgoing_syn_ee[outgoing_syn_ee.tag == 'spine']
outgoing_syn_onto_nonspines_ee = outgoing_syn_ee[outgoing_syn_ee.tag != 'spine']

In [8]:
def plot_single_neuron_wrap(ax, outgoing_syn_onto_spine, outgoing_syn_onto_nonspine,
                            circle_size=12, alpha_synapse=0.6):
    plot_single_neuron(ax=ax, sk_a=sk_axon, sk_d=sk_dent,
                       syn_spines=outgoing_syn_onto_spine, syn_nonspines=outgoing_syn_onto_nonspine,
                       circle_size=circle_size, alpha_synapse=alpha_synapse,
                       axon_color=ex_color, dend_color='black',
                       spiny_color=spiny_color, aspiny_color=aspiny_color)
    ax.tick_params(left=False, right=False, labelleft=False, labelbottom=False, bottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    y0, y1 = ax.get_ylim()
    y_span = abs(y1 - y0)
    crop_y_amount = y_span * 0.3
    if y0 > y1:
        ax.set_ylim(y0 - crop_y_amount, y1)
    else:
        ax.set_ylim(y0 + crop_y_amount, y1)
    ax.text(0.45, 0.01, '=', transform=ax.transAxes, ha='right', va='top',
            fontsize=22, color='black', clip_on=False)
    x0, x1 = ax.get_xlim()
    ax.set_xlim(x0, 1000)
    ax.text(1.05, 0.3, '=', transform=ax.transAxes, rotation=90,
            ha='right', va='top', fontsize=22, color='black', clip_on=False)

In [9]:
debug = False
fig = plt.figure(figsize=(24, 15), dpi=600)
gs = fig.add_gridspec(3, 4, width_ratios=[0.5, 1.5, 1, 1], wspace=0.2, hspace=0.4)

ax_cyl1 = fig.add_subplot(gs[0, 0], projection='3d')
ax_cyl2 = fig.add_subplot(gs[1, 0], projection='3d')
ax_cyl3 = fig.add_subplot(gs[2, 0], projection='3d')

plot_nested_cylinders(ax_cyl1, show_outer=True, ex_color=ex_color, inh_color=inh_color)
plot_nested_cylinders(ax_cyl2, show_outer=False, ex_color=ex_color, inh_color=inh_color)
plot_nested_cylinders(ax_cyl3, show_outer=False, ex_color=ex_color, inh_color=None)
ax_cyl1.set_title('Whole volume')
ax_cyl2.set_title('Micro column')
ax_cyl3.set_title('Micro column')
plot_legend_EI_clf_type(ax=ax_cyl1, markersize=4)

ax_hist1 = fig.add_subplot(gs[0, 1])
ax_hist2 = fig.add_subplot(gs[1, 1])
ax_hist3 = fig.add_subplot(gs[2, 1])

for i, (ax, outgoing_syn) in enumerate(zip(
        [ax_hist1, ax_hist2, ax_hist3],
        [ex_outgoing_syn_with_tags, ex_syn_tags, ee_syn_tags])):
    ax.set_title(titles[i])
    plot_raw_bin(outgoing_syn, ax=ax, is_ei=(i == 1),
                 spiny_color=spiny_color, aspiny_color=aspiny_color,
                 spiny_color_ei=spiny_color_ei, aspiny_color_ei=aspiny_color_ei)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_xlabel('Axonal path length from soma (\u03bcm)')
    ax.grid(True, axis='y', linestyle='--', linewidth=0.5, alpha=0.7)
    ax.legend(frameon=False, loc='upper right')

ax_pop1 = fig.add_subplot(gs[0, 2])
ax_pop2 = fig.add_subplot(gs[1, 2])
ax_pop3 = fig.add_subplot(gs[2, 2])

fit_pop(ex_outgoing_syn_with_tags, num_pop_bins=160, max_distance=1200, ax=ax_pop1, log=False, pop_color=ex_color)
fit_pop(ex_syn_tags, num_pop_bins=50, max_distance=800, ax=ax_pop2, log=False, pop_color=ex_color)
fit_pop(ee_syn_tags, num_pop_bins=50, max_distance=600, ax=ax_pop3, log=False, pop_color=ex_color)

ax_single1 = fig.add_subplot(gs[0, 3], sharey=ax_pop1)
ax_single2 = fig.add_subplot(gs[1, 3], sharey=ax_pop2)
ax_single3 = fig.add_subplot(gs[2, 3], sharey=ax_pop3)

fit_single_wrap(outgoing_syn_df=ex_outgoing_syn_with_tags, single_bin_width=16, min_synapses_per_bin=1,
                max_distance=1200, ax=ax_single1, root_id=node_id, color=spiny_color,
                log=False, axon_label='Neuron in Fig. 3A')
fit_single_wrap(outgoing_syn_df=ex_syn_tags, single_bin_width=6, min_synapses_per_bin=1,
                max_distance=800, ax=ax_single2, root_id=node_id, color=spiny_color,
                log=False, axon_label='Neuron in Fig. 3A')
fit_single_wrap(outgoing_syn_df=ee_syn_tags, single_bin_width=6, min_synapses_per_bin=1,
                max_distance=600, ax=ax_single3, root_id=node_id, color=spiny_color,
                log=False, axon_label='Neuron in Fig. 3A')

for ax in [ax_pop1, ax_pop2, ax_pop3, ax_single1, ax_single2, ax_single3]:
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0, symbol=''))
    ax.spines[['top', 'right']].set_visible(False)
    if ax in [ax_pop1, ax_pop2, ax_pop3]:
        ax.set_ylabel('% of output synapses\n on target spines')
        ax.legend(frameon=False, loc='lower right')
    if ax in [ax_single1, ax_single2, ax_single3]:
        ax.tick_params(labelleft=False)
        ax.legend(frameon=False, loc='lower left')
    ax.set_xlabel('Axonal path length from soma (\u03bcm)')
    ax.grid(True, axis='y', linestyle='--', linewidth=0.5, alpha=0.7)
    ax.set_ylim(bottom=0)
    ax.set_xlim(left=0)

ax_single2.set_xlim(0, 800)
ax_single3.set_xlim(0, 600)

ax_inset_neuron = ax_single1.inset_axes([0.5, -0.05, 0.5, 0.95])
ax_inset_neuron.set_facecolor((1, 1, 1, 0.65))
plot_single_neuron_wrap(ax=ax_inset_neuron,
                        outgoing_syn_onto_spine=outgoing_syn_onto_spines_outside,
                        outgoing_syn_onto_nonspine=outgoing_syn_onto_nonspines_outside,
                        circle_size=3, alpha_synapse=0.65)
ax_single1.set_zorder(2)
ax_inset_neuron.set_zorder(1)

ax_inset_neuron2 = ax_single2.inset_axes([0.5, -0.05, 0.5, 0.95])
ax_inset_neuron2.set_facecolor((1, 1, 1, 0.65))
plot_single_neuron_wrap(ax=ax_inset_neuron2,
                        outgoing_syn_onto_spine=outgoing_syn_onto_spines_all,
                        outgoing_syn_onto_nonspine=outgoing_syn_onto_nonspines_all,
                        circle_size=6, alpha_synapse=0.75)

ax_inset_neuron3 = ax_single3.inset_axes([0.5, -0.05, 0.5, 0.95])
ax_inset_neuron3.set_facecolor((1, 1, 1, 0.65))
plot_single_neuron_wrap(ax=ax_inset_neuron3,
                        outgoing_syn_onto_spine=outgoing_syn_onto_spines_ee,
                        outgoing_syn_onto_nonspine=outgoing_syn_onto_nonspines_ee,
                        circle_size=6, alpha_synapse=0.75)

add_panel_label(ax_cyl1, 'A', xy=(-0.01, 1.2))
add_panel_label(ax_cyl2, 'B', xy=(-0.01, 1.2))
add_panel_label(ax_cyl3, 'C', xy=(-0.01, 1.2))
plt.savefig('fig_s4.pdf', format='pdf', bbox_inches='tight')

Global spine fraction for neuron 864691135617152361: 0.7436
Global spine fraction for neuron 864691135617152361: 0.7708
Global spine fraction for neuron 864691135617152361: 0.9467
